<a href="https://colab.research.google.com/github/tiran543/Statistical-Learning-e23094/blob/main/Copy_of_GPR_LR_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [1]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

100%|██████████| 6.22k/6.22k [00:00<00:00, 10.3MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/elikplim/eergy-efficiency-dataset/versions/1


In [2]:
import os
import pandas as pd  # <--- Add this line right here!

print(f"Listing contents of: {path}")
!ls {path}
df2 = pd.read_csv(path + "/ENB2012_data.csv")

Listing contents of: /root/.cache/kagglehub/datasets/elikplim/eergy-efficiency-dataset/versions/1
ENB2012_data.csv


In [3]:
df2.head()

,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


In [4]:
# Extract the features (inputs)
X = df2[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']]

# Extract the targets (outputs) - modeling both as requested
y = df2[['Y1', 'Y2']]

# Print the shapes to verify
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (768, 8)
Shape of y: (768, 2)


In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Define the kernel (this tells the model how to measure similarity between data points)
# We use a Constant Kernel multiplied by an RBF (Radial Basis Function) kernel
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))

# Initialize the Gaussian Process Regressor
# We set n_restarts_optimizer to help the model find the best kernel parameters
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

# Fit the model to your data (X and y)
print("Training the GPR model... (this might take a few seconds)")
gpr.fit(X, y)

print("Model training complete!")
print("Optimized Kernel:", gpr.kernel_)

Training the GPR model... (this might take a few seconds)
Model training complete!
Optimized Kernel: 18.1**2 * RBF(length_scale=0.899)


In [6]:
# Check the R-squared score of the model
score = gpr.score(X, y)
print(f"Model R^2 Score: {score:.4f}")

Model R^2 Score: 1.0000


### Gaussian Process Regression: Conclusions

**R-Squared Score Analysis:** The model achieved an $R^2$ score of 1.0000. While this technically indicates a "perfect" fit on the training data, in practice, it is a massive red flag for **overfitting**. Because we trained and tested on the exact same data without a train/test split, the model has essentially memorized the training dataset points. It is highly unlikely to generalize well to unseen building data.

**Modeling Heating and Cooling Together:**
While the `GaussianProcessRegressor` is technically capable of multi-output regression, modeling heating and cooling loads together under a single parameter Gaussian Process is generally not optimal. Heating and cooling loads often have inverse or completely distinct relationships with building features (e.g., increasing a certain glazing area might drastically increase the cooling load while decreasing the heating load). By forcing them to share a single covariance function (kernel) and identical hyperparameters, the model fails to capture the unique, distinct physical behaviors of each separate load. Training two individual GP models would likely yield more robust, interpretable results.

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [7]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

100%|██████████| 347k/347k [00:00<00:00, 66.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/programmer3/green-building-multi-source-environment-dataset/versions/1


In [8]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2 = pd.read_csv(path + "/green_building_dataset.csv")

Listing contents of: /root/.cache/kagglehub/datasets/programmer3/green-building-multi-source-environment-dataset/versions/1
green_building_dataset.csv


In [9]:
# Look at the first 5 rows
display(df2.head())

# Print all the column names to see our options
print(df2.columns)

,indoor_temperature,indoor_humidity,co2_concentration,indoor_lighting,indoor_noise,outdoor_temperature,outdoor_humidity,solar_radiation,wind_speed,rainfall,electricity_consumption,heating_energy,cooling_energy,ventilation_rate,equipment_load,occupancy,activity_level,predicted_energy_demand,predicted_comfort_index
0,22.494481,43.624167,554.345944,432.115959,30.958646,24.443784,22.670752,540.768233,0.333310,47.820981,34.276401,18.919498,21.254016,327.046999,29.348868,26,0,39.936909,0.234932
1,29.408572,32.868476,466.383802,221.965186,68.624892,-1.398534,50.087239,699.959413,5.054747,43.364194,23.378548,17.726091,18.000948,144.862778,26.654788,7,0,24.985061,0.000000
2,26.783927,46.385156,1850.558681,566.559664,38.547245,5.904842,24.415262,828.108509,12.980562,36.379122,2.785345,19.930580,39.099193,493.647357,24.212357,43,1,39.675344,0.000000
3,25.183902,42.448700,663.712464,201.348306,32.195231,29.815571,75.240077,791.541006,0.652026,3.769213,45.925508,17.374061,37.267514,475.091197,6.281035,3,1,52.678350,0.000000
4,19.872224,57.084826,1705.062755,940.588677,62.684935,18.790863,57.069417,882.605624,6.433936,2.452494,49.016457,21.653203,45.261246,287.220492,4.693055,20,3,48.824527,0.000000


Index(['indoor_temperature', 'indoor_humidity', 'co2_concentration',
       'indoor_lighting', 'indoor_noise', 'outdoor_temperature',
       'outdoor_humidity', 'solar_radiation', 'wind_speed', 'rainfall',
       'electricity_consumption', 'heating_energy', 'cooling_energy',
       'ventilation_rate', 'equipment_load', 'occupancy', 'activity_level',
       'predicted_energy_demand', 'predicted_comfort_index'],
      dtype='object')


In [12]:
from sklearn.linear_model import LinearRegression

# 1. Choose your input features (X) - updated to better parameters
X = df2[['heating_energy', 'cooling_energy', 'electricity_consumption']]

# 2. Set the target variable (y) as defined by the assignment
y = df2['predicted_energy_demand']

# 3. Initialize and train the Linear Regression model
lr = LinearRegression()
lr.fit(X, y)

# 4. Check how well it performed
score = lr.score(X, y)
print(f"Linear Regression R^2 Score: {score:.4f}")

Linear Regression R^2 Score: 0.3791


### Linear Regression: Justification and Results

**Parameter Justification:** I selected `heating_energy`, `cooling_energy`, and `electricity_consumption` as the input features ($X$). The logical basis for this choice is that these three parameters represent the primary, active components of a building's energy usage. A building's overall `predicted_energy_demand` should fundamentally be a composite of how much it costs to heat it, cool it, and power its baseline electrical systems.

**Discussion of Results:** The model yielded an $R^2$ score of 0.3791. This indicates that our linear model only explains about ~38% of the variance in the predicted energy demand, which is quite low. This poor performance suggests two main conclusions:
1. **Missing Context:** We are likely missing critical features. Factors like `outdoor_temperature`, `occupancy`, or `solar_radiation` probably play a massive role in the algorithm that generated the target `predicted_energy_demand`.
2. **Non-Linearity:** The true relationship between energy consumption components and the overall predicted demand may not be strictly linear. A simple linear regression model is likely too basic to capture the complex, interacting variables of building physics.